# 07 — Temporal Comparison

In this notebook we will:
1. Understand why temporal queries break simple RAG
2. Build a multi-retrieval strategy (one query per quarter)
3. Create a comparison prompt that synthesizes across time periods
4. Build a temporal RAG chain end-to-end

### The problem
A question like *"How has revenue changed across the last 3 quarters?"* can't be answered by a single similarity search. The retriever would return the 4 most similar chunks overall, which might all come from the same quarter.

### The solution
Retrieve **separately per quarter**, then give the LLM all the contexts and ask it to compare.

```
Q3-2025 context: "Revenue was $94B, up 10%..."
Q4-2025 context: "Revenue was $102.5B, up 8%..."
Q1-2026 context: "Revenue was $69.6B, up 13%..."
         ↓
LLM: "Revenue grew from $94B to $102.5B, then..." 
```

## Setup

In [1]:
import sys
sys.path.insert(0, "..")

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv("../.env")

True

In [2]:
# ChromaDB
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(
    name="earnings_calls",
    embedding_function=embedding_fn,
)

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

print(f"Collection: {collection.count()} documents")

# What quarters do we have?
all_meta = collection.get(include=["metadatas"])
quarters = sorted(set(m["quarter"] for m in all_meta["metadatas"]))
print(f"Quarters available: {quarters}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Collection: 176 documents
Quarters available: ['Q1-2026', 'Q3-2025', 'Q4-2025']


## Step 1: Why single retrieval fails for temporal queries

Let's see what happens when we ask a cross-quarter question with a standard retrieval.

In [3]:
# Standard retrieval — single query, no quarter filter
results = collection.query(
    query_texts=["How has revenue changed across quarters?"],
    n_results=4,
)

print("Standard retrieval for: 'How has revenue changed across quarters?'\n")
for i in range(len(results["documents"][0])):
    meta = results["metadatas"][0][i]
    print(f"  Result {i+1}: {meta['quarter']} — {meta['speaker']} (dist: {results['distances'][0][i]:.3f})")

result_quarters = [results["metadatas"][0][i]["quarter"] for i in range(len(results["documents"][0]))]
print(f"\nQuarters in results: {result_quarters}")
print(f"Unique quarters: {set(result_quarters)}")
print("\n^ Likely missing some quarters — that's why we need per-quarter retrieval.")

Standard retrieval for: 'How has revenue changed across quarters?'

  Result 1: Q4-2025 — Kevan Parekh (dist: 0.415)
  Result 2: Q3-2025 — Benjamin Alexander Reitzes (dist: 0.435)
  Result 3: Q3-2025 — Timothy D. Cook (dist: 0.452)
  Result 4: Q4-2025 — Wamsi Mohan (dist: 0.468)

Quarters in results: ['Q4-2025', 'Q3-2025', 'Q3-2025', 'Q4-2025']
Unique quarters: {'Q3-2025', 'Q4-2025'}

^ Likely missing some quarters — that's why we need per-quarter retrieval.


## Step 2: Per-quarter retrieval

We retrieve the most relevant chunks for each quarter separately, guaranteeing coverage.

In [4]:
def retrieve_per_quarter(
    query: str,
    quarters: list[str],
    n_per_quarter: int = 2,
    company: str = None,
) -> dict[str, list[dict]]:
    """Retrieve top chunks for each quarter separately.
    
    Returns a dict: {quarter: [list of {text, speaker, role}]}
    """
    results_by_quarter = {}
    
    for quarter in quarters:
        where_filter = {"quarter": quarter}
        if company:
            where_filter = {
                "$and": [
                    {"quarter": quarter},
                    {"company": company},
                ]
            }
        
        results = collection.query(
            query_texts=[query],
            n_results=n_per_quarter,
            where=where_filter,
        )
        
        quarter_docs = []
        for i in range(len(results["documents"][0])):
            quarter_docs.append({
                "text": results["documents"][0][i],
                "speaker": results["metadatas"][0][i].get("speaker", "?"),
                "role": results["metadatas"][0][i].get("role", ""),
                "distance": results["distances"][0][i],
            })
        results_by_quarter[quarter] = quarter_docs
    
    return results_by_quarter

# Test it
results = retrieve_per_quarter("revenue results", quarters, company="AAPL")
for q, docs in results.items():
    print(f"\n{q}:")
    for d in docs:
        print(f"  [{d['speaker']}] {d['text'][:100]}...")


Q1-2026:

Q3-2025:
  [Kevan Parekh] Thanks, Tim, and good afternoon, everyone. Our revenue of $94 billion was up 10% year-over-year and ...
  [Timothy D. Cook] Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Today, we are proud ...

Q4-2025:
  [Timothy Cook] Thank you, Suhasini. Good afternoon, everyone, and thanks for joining the call. Today, Apple is prou...
  [Kevan Parekh] Thanks, Tim, and good afternoon, everyone. Our revenue of $102.5 billion was up 8% year-over-year an...


## Step 3: Build the temporal context

We format the per-quarter results into a structured context that makes it easy for the LLM to compare across time.

In [5]:
def build_temporal_context(results_by_quarter: dict[str, list[dict]]) -> str:
    """Format per-quarter results into structured context for the LLM."""
    sections = []
    for quarter in sorted(results_by_quarter.keys()):
        docs = results_by_quarter[quarter]
        chunks = []
        for d in docs:
            speaker_info = d["speaker"]
            if d["role"]:
                speaker_info += f" ({d['role']})"
            chunks.append(f"[{speaker_info}]: {d['text']}")
        section_text = "\n\n".join(chunks)
        sections.append(f"=== {quarter} ===\n{section_text}")
    
    return "\n\n".join(sections)

# Preview
ctx = build_temporal_context(results)
print(ctx[:800] + "...")

=== Q1-2026 ===


=== Q3-2025 ===
[Kevan Parekh (Chief Financial Officer)]: Thanks, Tim, and good afternoon, everyone. Our revenue of $94 billion was up 10% year-over-year and is a new June quarter record. We grew in every geographic segment and in the vast majority of the markets we track. Products revenue was $66.6 billion, up 8% year-over-year, driven by growth across iPhone and Mac. And thanks to our high levels of customer satisfaction and strong loyalty, our installed base of active devices reached another all-time high across all product categories and geographic segments. Services revenue was $27.4 billion, up 13% year-over-year and an all-time record. We saw strength across the world with double-digit growth in the majority of our markets.
Company gross margin was 46.5% at the hig...


## Step 4: Temporal comparison prompt

---
### Your turn!

Write a prompt template for temporal comparisons. It receives `{context}` (organized by quarter) and `{question}`.

Think about what makes a good temporal analysis — the LLM should identify trends, changes, and key differences across the time periods.

In [8]:
# -----------------------------------------------
# YOUR TURN: write the temporal comparison prompt
# -----------------------------------------------
#
# The context is organized by quarter with === Q#-YYYY === headers.
# The LLM should:
# - Analyze each quarter's data
# - Identify trends and changes over time
# - Be specific with numbers when available
# - Note when information is missing for a quarter

TEMPORAL_PROMPT_TEMPLATE = """
You are an analyst assistant that answers questions about finance using the earnings calls provided.
Use ONLY the provided context to answer. If the context doesn't contain 
the answer, say so. Always mention which company and quarter the information 
comes from.

The context is organized by quarter with === Q#-YYYY === headers. It is important to use all the provided context
to have a broader picture of the answer. 

Finally, summarize and compare the results to give a detailed analysis.


Context:
{context}

Question: {question}
"""

temporal_prompt = ChatPromptTemplate.from_template(TEMPORAL_PROMPT_TEMPLATE)
temporal_chain = temporal_prompt | llm | StrOutputParser()

print("Prompt variables:", temporal_prompt.input_variables)

Prompt variables: ['context', 'question']


### Validate

In [9]:
assert "context" in temporal_prompt.input_variables, "Must use {context}"
assert "question" in temporal_prompt.input_variables, "Must use {question}"
assert len(TEMPORAL_PROMPT_TEMPLATE.strip()) > 50, "Prompt seems too short"

print("All validations passed!")

All validations passed!


## Step 5: Full temporal RAG pipeline

In [10]:
def ask_temporal(
    question: str,
    quarters: list[str] = None,
    company: str = None,
    n_per_quarter: int = 2,
) -> str:
    """Full temporal RAG: retrieve per quarter, then compare."""
    if quarters is None:
        all_meta = collection.get(include=["metadatas"])
        quarters = sorted(set(m["quarter"] for m in all_meta["metadatas"]))
    
    results = retrieve_per_quarter(question, quarters, n_per_quarter, company)
    context = build_temporal_context(results)
    return temporal_chain.invoke({"context": context, "question": question})

print("Temporal RAG pipeline ready.")

Temporal RAG pipeline ready.


In [11]:
# Test 1: Revenue trend
q = "How has Apple's revenue changed across quarters?"
print(f"Q: {q}\n")
print(ask_temporal(q, company="AAPL"))

Q: How has Apple's revenue changed across quarters?

The information provided is from Apple and covers the Q3-2025 and Q4-2025 quarters. 

In Q3-2025, Apple reported a revenue record of $94 billion, up 10% from a year ago. 

In Q4-2025, Apple reported a revenue of $102.5 billion, up 8% from a year ago, which is a September quarter record.

Comparing the two quarters, Apple's revenue increased by $8.5 billion from Q3-2025 to Q4-2025. 

It's worth noting that there is no information provided about Q1-2026 and Q2-2025, so we cannot determine the full trend of Apple's revenue across all quarters. 

However, based on the provided context, we can see that Apple's revenue has been increasing, with Q4-2025 being the highest revenue reported. Apple expects its December quarter total company revenue to grow by 10% to 12% year-over-year, which would be its best quarter ever.


In [12]:
# Test 2: Margin guidance evolution
q = "How has gross margin guidance evolved over the last quarters?"
print(f"Q: {q}\n")
print(ask_temporal(q, company="AAPL"))

Q: How has gross margin guidance evolved over the last quarters?

The information provided is from Apple and covers the Q3-2025 and Q4-2025 quarters. 

In Q3-2025, the company gross margin was 46.5% at the high end of the guidance range and down 60 basis points sequentially. For the September quarter (Q4 is not explicitly mentioned but based on the context it seems the September quarter guidance is provided in the Q3-2025 section), the gross margin is expected to be between 46% and 47%, which includes the estimated impact of the $1.1 billion tariff-related costs.

In Q4-2025, the gross margin expectations for the December quarter imply an increase of 30 basis points sequentially. However, the exact gross margin percentage for the December quarter is not provided.

Comparing the results, we can see that the gross margin guidance has remained relatively stable, with a slight increase expected in the December quarter. The company has managed to maintain its gross margin despite the impact

In [13]:
# Test 3: Topic trend
q = "How has the discussion about AI and Apple Intelligence evolved across quarters?"
print(f"Q: {q}\n")
print(ask_temporal(q, company="AAPL"))

Q: How has the discussion about AI and Apple Intelligence evolved across quarters?

The discussion about AI and Apple Intelligence has evolved across quarters, specifically from Q3-2025 to Q4-2025, at Apple. 

In Q3-2025, Timothy D. Cook, the Chief Executive Officer, mentioned that the company is making good progress on a more personalized Siri and expects to release the features next year. He also stated that the company is significantly growing its investment in AI and reallocating a fair number of people to focus on AI features within the company. Additionally, Cook emphasized the importance of AI, calling it one of the most profound technologies of our lifetime, and expressed excitement about the company's road map ahead.

In Q4-2025, the discussion shifted to the impact of AI on consumer purchasing decisions and Apple's approach to AI capabilities. Richard Kramer asked if AI capabilities or features are a material purchase consideration for consumers, given the record sales levels

In [14]:
# Test 4: Specific quarters only
q = "Compare Apple's services revenue between Q3 and Q4 2025"
print(f"Q: {q}\n")
print(ask_temporal(q, quarters=["Q3-2025", "Q4-2025"], company="AAPL"))

Q: Compare Apple's services revenue between Q3 and Q4 2025

Apple's services revenue for Q3-2025 was $27.4 billion, up 13% from a year ago, and an all-time record. 

In Q4-2025, Apple's services revenue reached an all-time high of $28.8 billion, up 15% year-over-year. 

Comparing the two quarters, we can see that services revenue increased by $1.4 billion from Q3 to Q4, representing a 5.1% increase. This indicates a continued strong growth in Apple's services segment. 

(Company: Apple, Quarters: Q3-2025 and Q4-2025)


## Summary

In this notebook you learned:
- Standard RAG retrieval fails for temporal queries because it doesn't guarantee coverage across time periods
- Per-quarter retrieval ensures every time period is represented in the context
- Structured context (organized by quarter) helps the LLM produce better comparisons
- The temporal prompt needs to guide the LLM toward trends, changes, and specific numbers

### Phase 2 almost complete!

You now have three retrieval strategies:
1. **Simple semantic** (notebook 03) — find by meaning
2. **Self-query** (notebook 06) — automatic metadata filtering from natural language
3. **Temporal** (this notebook) — per-quarter retrieval for cross-time comparisons

**Next step:** Notebook 08 — basic evaluation to measure retrieval and answer quality.